<a href="https://colab.research.google.com/github/edwardsnj/pride-study-retrieval-cofest-2026/blob/main/notebooks/TextEmbedding%2BTFIDF_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [105]:
# files...
BASEURL = "https://edwardslab.bmcb.georgetown.edu/~nedwards/dropbox/6ItUS2tEdC/"
csvfile = "pride-embeddings-openai-3-small.csv"
fthfile = "pride-embeddings-openai-3-small.fth"

GITHUB = "https://raw.githubusercontent.com/EdwardsLabProjects/pride-study-retrieval-cofest-2026/refs/heads/main/data/"
trueposfile = "truepos.txt"
truenegfile = "trueneg.txt"

# download...
for f in [csvfile, fthfile]:
  !wget -nc {BASEURL+f}
for f in [trueposfile, truenegfile]:
  !wget -nc {GITHUB+f}

import numpy as np
import random

# RANDOM SEED
RANDOMSEED = None
if not RANDOMSEED:
  RANDOMSEED = random.randint(1,10000000)

print(f"Using random seed: {RANDOMSEED}")

# Seeds all scikit-learn functions that default to random_state=None
np.random.seed(RANDOMSEED)
random.seed(RANDOMSEED)


File ‘pride-embeddings-openai-3-small.csv’ already there; not retrieving.

File ‘pride-embeddings-openai-3-small.fth’ already there; not retrieving.

File ‘truepos.txt’ already there; not retrieving.

File ‘trueneg.txt’ already there; not retrieving.

Using random seed: 1065012


In [106]:
import pandas as pd

emb = pd.read_feather(fthfile)
md = pd.read_csv(csvfile)

tp = set(open(trueposfile).read().split())
tn = set(open(truenegfile).read().split())

# Keep only the ones that there are embeddings for
tp &= set(emb.columns)
tn &= set(emb.columns)

assert len(set(tp) & set(tn)) == 0, "TP and TN should not intersect!"

# Print out some details...
print(f"Embeddings: {emb.shape}")
print(f"True Pos: {len(tp)}")
print(f"True Neg: {len(tn)}")

Embeddings: (1536, 36696)
True Pos: 84
True Neg: 47


In [107]:
from sklearn.model_selection import train_test_split

def split_train_test(embeddings, seeds, neg_seeds, test_size=0.2, bgsize=25):
      seeds = list(set(seeds)&set(embeddings.columns))
      neg_seeds = list(set(neg_seeds)&set(embeddings.columns))
      bg = list(set(embeddings.columns)-set(seeds))
      nbgsel = int(round(len(seeds)*bgsize))
      if len(bg) < len(neg_seeds):
        raise ValueError("Not enough background embeddings to create selected background.")
      if len(bg) < nbgsel:
        raise ValueError("Not enough background embeddings to create selected background.")

      if test_size > 0.0:
        # 1. Split the original seed set
        pos_train_accs, pos_test_accs = train_test_split(
            seeds,
            test_size=test_size
        )
        have_test = True
      else:
        pos_train_accs = list(seeds)
        pos_test_accs = []
        have_test = False

      selected_accessions = list(seeds)
      train_accessions = list(pos_train_accs)
      test_accessions = list(pos_test_accs)

      num_train_samples = len(pos_train_accs)
      num_bg_train_samples = int(round(num_train_samples*bgsize))
      num_test_samples = len(pos_test_accs)
      num_bg_test_samples = nbgsel-num_bg_train_samples

      selbg = neg_seeds + list(random.sample(list(set(bg)-set(neg_seeds)),
                                             nbgsel-len(neg_seeds)))
      random.shuffle(selbg)
      seltrainbg = selbg[:num_bg_train_samples]
      seltestbg = selbg[num_bg_train_samples:]

      selected_accessions += selbg
      train_accessions += seltrainbg
      test_accessions += seltestbg
      train_y = [1]*num_train_samples + [0]*num_bg_train_samples
      test_y = [1]*num_test_samples + [0]*num_bg_test_samples

      return train_accessions, train_y, test_accessions, test_y

train_acc, train_y, test_acc, test_y = \
    split_train_test(emb, tp, tn,
                     test_size=0.2, bgsize=25)


In [108]:
from sklearn.feature_extraction.text import TfidfVectorizer

def create_tfidf_features(md_dataframe, train_accessions, train_y, test_accessions, positive_only=False,**kwargs):
    # Filter training accessions to include only 'true cases' (where train_y is 1)
    if positive_only:
      true_train_accessions = [acc for acc, y_val in zip(train_accessions, train_y) if y_val == 1]
    else:
      true_train_accessions = train_accessions

    # Prepare true case training data for TF-IDF fitting
    # Ensure the order of texts matches the order of true_train_accessions for correct indexing
    md_train_true_cases = md_dataframe[md_dataframe['prideacc'].isin(true_train_accessions)].set_index('prideacc').loc[true_train_accessions]
    train_texts_true_cases = md_train_true_cases['text']

    # Initialize TfidfVectorizer
    tfidf_vectorizer = TfidfVectorizer(**kwargs)

    # Fit TfidfVectorizer ONLY on the true cases of the training data
    tfidf_vectorizer.fit(train_texts_true_cases)

    # Prepare ALL training data for transformation
    md_train_all = md_dataframe[md_dataframe['prideacc'].isin(train_accessions)].set_index('prideacc').loc[train_accessions]
    train_texts_all = md_train_all['text']

    # Transform ALL training data using the fitted vectorizer
    tfidf_train_matrix = tfidf_vectorizer.transform(train_texts_all)

    # Create a DataFrame for training TF-IDF values
    tfidf_df_train = pd.DataFrame(
        tfidf_train_matrix.toarray(),
        index=train_accessions, # Index by pride accessions
        columns=tfidf_vectorizer.get_feature_names_out()
    )

    # Prepare testing data for TF-IDF transformation
    md_test = md_dataframe[md_dataframe['prideacc'].isin(test_accessions)].set_index('prideacc').loc[test_accessions]
    test_texts = md_test['text']

    # Apply the fitted TF-IDF model to test data (transform only)
    tfidf_test_matrix = tfidf_vectorizer.transform(test_texts)

    # Create a DataFrame for testing TF-IDF values
    tfidf_df_test = pd.DataFrame(
        tfidf_test_matrix.toarray(),
        index=test_accessions, # Index by pride accessions
        columns=tfidf_vectorizer.get_feature_names_out()
    )

    tfidf_df = pd.concat([tfidf_df_train, tfidf_df_test]).T

    return tfidf_df, tfidf_vectorizer

from sklearn.feature_extraction import text
english_stop_words = text.ENGLISH_STOP_WORDS

stop_words = list(set(list(english_stop_words) + """
""".split()))

extra_args = dict(positive_only=False,
                  min_df=1,
                  max_df=1.0,
                  max_features=None,
                  stop_words=stop_words)

tfidf,tfidf_model = create_tfidf_features(md, train_acc, train_y, test_acc, **extra_args)

In [109]:
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils import shuffle

def train_document_classifier(embeddings, tfidf, train_acc, train_y, test_acc, test_y, use_embed=True, use_tfidf=True, **kwargs):

    # Use np.hstack to concatenate features horizontally
    train_values = []
    if use_embed:
        train_values.append(embeddings[train_acc].values.T)
    if use_tfidf:
        train_values.append(tfidf[train_acc].values.T)
    train = np.hstack(train_values)
    if len(test_acc)>0:
        have_test = True
        # Use np.hstack for test features as well
        test_values = []
        if use_embed:
            test_values.append(embeddings[test_acc].values.T)
        if use_tfidf:
            test_values.append(tfidf[test_acc].values.T)
        test = np.hstack(test_values)

    # 4. Shuffle the datasets so labels are randomized (not all 1s followed by all 0s)
    X_train, y_train = shuffle(train, train_y)
    if have_test:
        X_test, y_test = shuffle(test, test_y)

    print(f"Training data shape: {X_train.shape} (Positives: {sum((y==1) for y in y_train)}, Negatives: {sum((y==0) for y in y_train)})")
    if have_test:
        print(f"Testing data shape: {X_test.shape} (Positives: {sum((y==1) for y in y_test)}, Negatives: {sum((y==0) for y in y_test)})")

    # 5. Initialize and train the Logistic Regression model
    model = LogisticRegression(max_iter=1000, **kwargs)
    model.fit(X_train, y_train)

    # 6. Evaluate the model on the withheld test set
    if not have_test:
        return model

    y_pred = model.predict(X_test)

    print("\n--- Model Evaluation ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print("\nClassification Report:")
    # target_names let us interpret the 1s and 0s easily in the terminal
    print(classification_report(y_test, y_pred, target_names=["Background (0)", "Seed-like (1)"]))

    return model

extra_args = dict(class_weight='balanced',C=10.0,
                  penalty='l1', solver='liblinear',
                  use_embed=True, use_tfidf=True)
trained_model = train_document_classifier(emb, tfidf,
                                          train_acc, train_y,
                                          test_acc, test_y,
                                          **extra_args)
print("NZ Coeffs:",sum(*[1*(xi!=0) for xi in trained_model.coef_]))

Training data shape: (1742, 37142) (Positives: 67, Negatives: 1675)
Testing data shape: (442, 37142) (Positives: 17, Negatives: 425)

--- Model Evaluation ---
Accuracy: 0.9842

Classification Report:
                precision    recall  f1-score   support

Background (0)       1.00      0.99      0.99       425
 Seed-like (1)       0.73      0.94      0.82        17

      accuracy                           0.98       442
     macro avg       0.86      0.96      0.91       442
  weighted avg       0.99      0.98      0.99       442

NZ Coeffs: 51


In [110]:
import pandas as pd

# Get the number of embedding features
num_embedding_features = emb.shape[0]

# Calculate significant embedding coefficients
embedding_coefficients = trained_model.coef_[0][:num_embedding_features]
significant_embedding_coeffs = np.sum(embedding_coefficients != 0)
print(f"\nNumber of semantic embedding non-zero coefficients: {significant_embedding_coeffs}")

# Get the TF-IDF feature names from the fitted vectorizer
tfidf_feature_names = tfidf_model.get_feature_names_out()

# The trained_model.coef_ is an array of coefficients for all features (embeddings + tfidf)
# We need the coefficients corresponding to the TF-IDF features
tfidf_coefficients = trained_model.coef_[0][num_embedding_features:]

# Calculate and print the number of non-zero TF-IDF coefficients
non_zero_tfidf_coeffs = np.sum(tfidf_coefficients != 0)
print(f"Number of TF-IDF dimensions with non-zero coefficients: {non_zero_tfidf_coeffs}")

# Create a DataFrame to link feature names with their coefficients
feature_importance_df = pd.DataFrame({
    'Feature': tfidf_feature_names,
    'Coefficient': tfidf_coefficients
})

# Sort by the absolute value of the coefficient to find the most important features
feature_importance_df['Abs_Coefficient'] = feature_importance_df['Coefficient'].abs()
most_important_features = feature_importance_df.sort_values(by='Abs_Coefficient', ascending=False)

# Display the top 20 most important TF-IDF features without the 'Abs_Coefficient' column
print("\nTop 20 Most Important TF-IDF Features (without Abs_Coefficient):")
display(most_important_features.drop(columns=['Abs_Coefficient']).head(20))



Number of semantic embedding non-zero coefficients: 1
Number of TF-IDF dimensions with non-zero coefficients: 50

Top 20 Most Important TF-IDF Features (without Abs_Coefficient):


,Feature,Coefficient
25643,pngase,102.580871
35523,β2ar,-41.133196
6213,asn,25.845002
15526,glycosylated,21.560271
10976,deamidation,19.025751
15519,glycosite,18.677552
15527,glycosylation,17.546566
32306,thermophilic,15.232531
22964,nmpo,14.048440
7860,bx,12.900397


In [111]:
import re

print("\nExample sentences for top TF-IDF features:")

# Get the list of top features
top_features_list = most_important_features['Feature'].head(20).tolist()

for feature in top_features_list:
    print(f"\nFeature: '{feature}'")
    # Find documents where the feature appears
    feature_pattern = f'\\b{feature}\\b' # Use word boundary to match whole words
    # Only consider documents that contain the feature
    relevant_docs = md[md['text'].str.contains(feature_pattern, case=False, na=False)]

    if not relevant_docs.empty:
        # Iterate through the first 5 relevant documents to get multiple examples,
        # ensuring each comes from a different PRIDE accession.
        # If there are fewer than 5 relevant_docs, it will iterate through all of them.
        for idx, row in relevant_docs.head(5).iterrows():
            pride_accession = row['prideacc']
            text_with_feature = row['text']

            # Use regex to split into sentences more robustly, considering periods and newlines
            sentences = re.split(r'(?<=\.)\s*|\n+', text_with_feature)

            found_sentence = None
            for sentence in sentences:
                # Clean the sentence: replace multiple spaces/newlines with single space, then strip
                cleaned_part = ' '.join(sentence.split()).strip()
                if feature.lower() in cleaned_part.lower() and cleaned_part: # Ensure cleaned_part is not empty
                    found_sentence = cleaned_part
                    break

            if found_sentence:
                # Ensure it ends with a period if it doesn't already, and is a single line
                if not found_sentence.endswith('.'):
                    found_sentence += '.'
                print(f"  {pride_accession} - {found_sentence}")
            else:
                # Fallback: if no specific sentence found, take first 200 chars, clean, and make it one line
                cleaned_text_slice = ' '.join(text_with_feature[:200].split()).strip()
                if not cleaned_text_slice.endswith('.'):
                    cleaned_text_slice += '...'
                print(f"  {pride_accession} - {cleaned_text_slice}")
    else:
        print("  No example document found for this feature.")


Example sentences for top TF-IDF features:

Feature: 'pngase'
  PXD001432 - , Germany) and then incubated with 20 µl 1x G7 buffer containing 500 U glyerol-free PNGase F (NEB) at 37 °C in a thermomixer for 6 h to release the glycopeptides from the beads.
  PXD010911 - 1 μL (500 units) of PNGase F was added to one tube of peptides in each pair of aliquots; the second tube was not treated with the glycosidase and served as the control.
  PXD005726 - After an initial incubation at 37 °C for 24 h, the samples were speed-vac dried and PNGase F (1:200) was added together with 18O water to identify N-linked glycosylation sites by isotopic mass-differences in the MS analysis.
  PXD003196 - Tissue homogenization, trypsin digestion, oxidation, hydrazide capture, PNGase F release, c18 cleanup, MS.
  PXD002849 - Lectin based Glycopeptide enrichment strategy followed by PNGase catalyzed deglycosylation reaction was employed to assign potential N-linked glycosylation sites.

Feature: 'β2ar'
  PXD006